In [37]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

os.environ["CENSUS_API_KEY"] = "b1278b466e026ad679a62a03dd4b044bb8b36e7c"
CENSUS_API_KEY = os.getenv("CENSUS_API_KEY")
print("Key loaded:", CENSUS_API_KEY is not None)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

# -----------------------------
# Paths / config
# -----------------------------
DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
OUT_DIR = DATA_DIR / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Update this path if needed
IGS_PATH = r"C:\Users\jabba\Desktop\Code\machine_learning\AUC_mastercard_challenge\src\Inclusive_Growth_Score_Data_Export_25-02-2026_134202.csv"

# Tract selection for focused EDA / testing
TARGET_TRACTS = None
# Example:
# TARGET_TRACTS = ["13089023301", "13089023405", "13089023302"]

# If TARGET_TRACTS is set, the code will infer state FIPS from those tracts
SAVE_INTERMEDIATE = True

# ACS 5-year currently available through 2024
ACS_MAX_AVAILABLE_YEAR = 2024
FILL_2025_WITH_2024_ACS = False  # optional, keep False by default

Key loaded: True


### Cell 2 — IGS load / clean helpers

In [38]:
IGS_STRING_COLS = {
    "County",
    "State",
    "BENCHMARK",
    "Census Tract FIPS code",
    "geoid",
}

IGS_ALIAS_MAP = {
    "Inclusive Growth Score": "igs_total",
    "Growth": "igs_growth",
    "Inclusion": "igs_inclusion",
    "Place": "igs_place",
    "Economy": "igs_economy",
    "Community": "igs_community",
    "Place Growth": "igs_place_growth",
    "Place Inclusion": "igs_place_inclusion",
    "Economy Growth": "igs_economy_growth",
    "Economy Inclusion": "igs_economy_inclusion",
    "Community Growth": "igs_community_growth",
    "Community Inclusion": "igs_community_inclusion",
}


def dedupe_columns(cols):
    seen = {}
    out = []
    for c in cols:
        c = str(c).strip()
        if c not in seen:
            seen[c] = 0
            out.append(c)
        else:
            seen[c] += 1
            out.append(f"{c}.{seen[c]}")
    return out


def _find_header_row(df_raw, required_tokens=("Census Tract FIPS", "Year")):
    tokens = [t.lower() for t in required_tokens]
    for i in range(min(len(df_raw), 50)):
        row = df_raw.iloc[i].astype(str).str.lower().tolist()
        if all(any(tok in cell for cell in row) for tok in tokens):
            return i
    raise ValueError("Could not find the real IGS header row.")


def normalize_geoid_series(series: pd.Series) -> pd.Series:
    s = series.astype("string").str.strip()

    numeric = pd.to_numeric(s, errors="coerce")
    numeric_mask = numeric.notna()

    s = s.copy()
    s.loc[numeric_mask] = numeric.loc[numeric_mask].astype("Int64").astype("string")

    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.str.replace(r"\D", "", regex=True)
    s = s.str.zfill(11)
    s = s.where(s.str.fullmatch(r"\d{11}"), pd.NA)

    return s


def filter_tracts(df: pd.DataFrame, tracts=None, geoid_col="geoid") -> pd.DataFrame:
    if not tracts:
        return df.copy()

    tract_set = {str(t).zfill(11) for t in tracts}
    out = df[df[geoid_col].astype("string").isin(tract_set)].copy()
    return out


def safe_save_parquet(df: pd.DataFrame, path: Path):
    out = df.copy()

    for c in out.columns:
        if out[c].dtype == "object":
            out[c] = out[c].astype("string")

    out.to_parquet(path, index=False, engine="pyarrow")


def load_igs_any(path: str) -> pd.DataFrame:
    p = Path(path)

    if p.suffix.lower() in [".xlsx", ".xls"]:
        raw = pd.read_excel(p, header=None)
    else:
        raw = pd.read_csv(p, header=None, low_memory=False)

    hdr_i = _find_header_row(raw, required_tokens=("Census Tract FIPS", "Year"))
    header = dedupe_columns(raw.iloc[hdr_i].tolist())

    df = raw.iloc[hdr_i + 1 :].copy()
    df.columns = header
    df = df.dropna(how="all").reset_index(drop=True)

    # Drop any duplicated header rows that still appear in the body
    if "Census Tract FIPS code" in df.columns:
        df = df[
            df["Census Tract FIPS code"].astype(str).str.strip().str.lower() != "census tract fips code"
        ].copy()

    # Normalize GEOID + year
    df["geoid"] = normalize_geoid_series(df["Census Tract FIPS code"])
    df["year"] = pd.to_numeric(df["Year"], errors="coerce")

    # Convert non-ID columns to numeric where possible
    for c in df.columns:
        if c not in IGS_STRING_COLS:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # Friendly aliases for easier modeling later
    for old_col, new_col in IGS_ALIAS_MAP.items():
        if old_col in df.columns:
            df[new_col] = pd.to_numeric(df[old_col], errors="coerce")

    if "Is an Opportunity Zone" in df.columns:
        df["is_opp_zone"] = pd.to_numeric(df["Is an Opportunity Zone"], errors="coerce")

    if "URBAN CODE" in df.columns:
        df["urban_code"] = pd.to_numeric(df["URBAN CODE"], errors="coerce")

    if "BENCHMARK" in df.columns:
        df["benchmark"] = df["BENCHMARK"].astype("string")

    # Final cleanup
    df = df.dropna(subset=["geoid", "year"]).copy()
    df["year"] = df["year"].astype(int)
    df = df.sort_values(["geoid", "year"]).reset_index(drop=True)

    return df

### Cell 3 — Load and save clean IGS

In [39]:
igs = load_igs_any(IGS_PATH)
igs = filter_tracts(igs, TARGET_TRACTS)

if SAVE_INTERMEDIATE:
    safe_save_parquet(igs, OUT_DIR / "igs_clean.parquet")

print("IGS shape:", igs.shape)
print("IGS years:", sorted(igs["year"].dropna().unique().tolist())[:10], "...", sorted(igs["year"].dropna().unique().tolist())[-3:])
print()
print(igs[["geoid", "year", "igs_total"]].head())
print()
print(igs[["igs_total", "igs_place", "igs_economy", "igs_community"]].describe())
print()
print("Top missingness:")
print(igs.isna().mean().sort_values(ascending=False).head(20))

IGS shape: (765288, 87)
IGS years: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025] ... [2023, 2024, 2025]

         geoid  year  igs_total
0  01001020100  2017       47.0
1  01001020100  2018       52.0
2  01001020100  2019       46.0
3  01001020100  2020       46.0
4  01001020100  2021       38.0

           igs_total      igs_place    igs_economy  igs_community
count  757582.000000  761912.000000  764294.000000  762407.000000
mean       50.052242      49.848921      50.328149      50.148915
std        10.252555      12.453479      13.532944      13.744805
min         0.000000       0.000000       0.000000       0.000000
25%        44.000000      42.000000      42.000000      40.000000
50%        50.000000      50.000000      51.500000      50.000000
75%        57.900000      58.000000      60.000000      60.000000
max        84.000000     100.000000      94.000000     100.000000

Top missingness:
Is an Opportunity Zone                      1.000000
Spend Growth Base, %        

### Cell 4 — ACS variables to pull

In [40]:
ACS_VAR_GROUPS = {
    "population_age": [
        "B01001_001E",  # total pop
        "B01001_003E", "B01001_004E", "B01001_005E", "B01001_006E",  # male under 18
        "B01001_027E", "B01001_028E", "B01001_029E", "B01001_030E",  # female under 18
        "B01001_020E", "B01001_021E", "B01001_022E", "B01001_023E", "B01001_024E", "B01001_025E",  # male 65+
        "B01001_044E", "B01001_045E", "B01001_046E", "B01001_047E", "B01001_048E", "B01001_049E",  # female 65+
    ],
    "race": [
        "B02001_001E",
        "B02001_002E",
        "B02001_003E",
        "B02001_005E",
        "B02001_008E",
    ],
    "education": [
        "B15003_001E",
        "B15003_022E", "B15003_023E", "B15003_024E", "B15003_025E",
    ],
    "employment": [
        "B23025_001E",
        "B23025_002E",
        "B23025_003E",
        "B23025_005E",
    ],
    "income_poverty": [
        "B19013_001E",  # median household income
        "B19301_001E",  # per-capita income
        "B19083_001E",  # gini
        "B17001_001E",  # poverty universe
        "B17001_002E",  # below poverty
    ],
    "commute": [
        "B08303_001E",
        "B08303_002E", "B08303_003E", "B08303_004E", "B08303_005E",
        "B08303_006E", "B08303_007E", "B08303_008E",
    ],
    "internet": [
        "B28002_001E",
        "B28002_002E",
    ],
    "housing": [
        "B25002_001E", "B25002_002E", "B25002_003E",   # total / occupied / vacant
        "B25003_001E", "B25003_002E", "B25003_003E",   # tenure
        "B25064_001E",                                  # median gross rent
        "B25077_001E",                                  # median home value
    ],
    "rent_burden": [
        "B25070_001E",
        "B25070_002E", "B25070_003E", "B25070_004E", "B25070_005E", "B25070_006E",
    ],
    "owner_cost_burden": [
        "B25091_001E",
        "B25091_003E", "B25091_004E", "B25091_005E", "B25091_006E", "B25091_007E",
        "B25091_014E", "B25091_015E", "B25091_016E", "B25091_017E", "B25091_018E",
    ],
    "early_education": [
        "B14003_004E", "B14003_013E", "B14003_032E", "B14003_041E",
    ],
    "health_insurance": [
        "B27001_001E",
        "B27001_005E", "B27001_008E", "B27001_011E", "B27001_014E", "B27001_017E",
        "B27001_020E", "B27001_023E", "B27001_026E", "B27001_029E",
        "B27001_033E", "B27001_036E", "B27001_039E", "B27001_042E", "B27001_045E",
        "B27001_048E", "B27001_051E", "B27001_054E", "B27001_057E",
    ],
}

ACS_VARS = sorted({v for group in ACS_VAR_GROUPS.values() for v in group})
print("ACS variable count:", len(ACS_VARS))

ACS variable count: 98


### Cell 5 — ACS API helpers

In [41]:
def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i : i + n]


def infer_state_fips_from_tracts(tracts):
    if not tracts:
        return None
    return sorted({str(t).zfill(11)[:2] for t in tracts})


def resolve_acs_years(igs_years, max_available_year=ACS_MAX_AVAILABLE_YEAR):
    igs_years = sorted({int(y) for y in igs_years if pd.notna(y)})
    acs_years = [y for y in igs_years if y <= max_available_year]
    return acs_years


def get_state_fips(year: int, api_key: str) -> list[str]:
    url = f"https://api.census.gov/data/{year}/acs/acs5"
    params = {"get": "NAME", "for": "state:*", "key": api_key}

    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()

    data = r.json()
    header, rows = data[0], data[1:]
    df = pd.DataFrame(rows, columns=header)

    return sorted(df["state"].unique().tolist())


def census_get_tracts(year: int, state_fips: str, vars_: list[str], api_key: str) -> pd.DataFrame:
    base = f"https://api.census.gov/data/{year}/acs/acs5"
    get_str = "NAME," + ",".join(vars_)

    params = [
        ("get", get_str),
        ("for", "tract:*"),
        ("in", f"state:{state_fips}"),
        ("in", "county:*"),
        ("key", api_key),
    ]

    r = requests.get(base, params=params, timeout=120)
    r.raise_for_status()

    data = r.json()
    header, rows = data[0], data[1:]
    df = pd.DataFrame(rows, columns=header)
    df["year"] = year

    return df


def to_numeric_safe(df: pd.DataFrame, exclude=("NAME", "state", "county", "tract", "year")) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        if c in exclude:
            continue
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out

ACS_SENTINELS = {-666666666, -333333333, -222222222}

def clean_acs_values(df: pd.DataFrame, exclude=("NAME", "state", "county", "tract", "year")):
    out = df.copy()
    for c in out.columns:
        if c in exclude:
            continue
        out[c] = pd.to_numeric(out[c], errors="coerce")
        out[c] = out[c].replace(list(ACS_SENTINELS), np.nan)
    return out
    

def download_acs_all(
    years: list[int],
    api_key: str,
    state_fips_filter: list[str] | None = None,
    max_vars_per_call: int = 45,
    sleep_seconds: float = 0.25,
) -> pd.DataFrame:
    if not api_key:
        raise ValueError("Missing CENSUS_API_KEY. Put it in your environment before running ACS pulls.")

    all_parts = []

    for year in years:
        year_out = RAW_DIR / f"acs5_tract_{year}.parquet"

        if year_out.exists():
            print(f"[cache] {year_out.name}")
            df_year = pd.read_parquet(year_out)
            all_parts.append(df_year)
            continue

        states = state_fips_filter if state_fips_filter else get_state_fips(year, api_key)
        print(f"[download] year={year} | states={len(states)}")

        state_frames = []

        for st in states:
            merged_state = None

            for var_chunk in chunks(ACS_VARS, max_vars_per_call):
                df_chunk = census_get_tracts(year, st, var_chunk, api_key)
                df_chunk = clean_acs_values(df_chunk)

                keys = ["state", "county", "tract", "NAME", "year"]
                if merged_state is None:
                    merged_state = df_chunk
                else:
                    merged_state = merged_state.merge(df_chunk, on=keys, how="outer")

                time.sleep(sleep_seconds)

            state_frames.append(merged_state)

        df_year = pd.concat(state_frames, ignore_index=True)

        if SAVE_INTERMEDIATE:
            safe_save_parquet(df_year, year_out)

        all_parts.append(df_year)

    acs = pd.concat(all_parts, ignore_index=True)
    return acs

### Cell 6 — ACS feature engineering

In [42]:
def safe_div(n, d):
    n = np.asarray(n, dtype="float64")
    d = np.asarray(d, dtype="float64")

    out = np.full_like(n, np.nan, dtype="float64")
    valid = (~np.isnan(d)) & (d != 0)
    np.divide(n, d, out=out, where=valid)
    return out
    
def make_features(acs: pd.DataFrame) -> pd.DataFrame:
    acs = acs.copy()

    acs["state"] = acs["state"].astype(str).str.zfill(2)
    acs["county"] = acs["county"].astype(str).str.zfill(3)
    acs["tract"] = acs["tract"].astype(str).str.zfill(6)
    acs["geoid"] = acs["state"] + acs["county"] + acs["tract"]

    out = pd.DataFrame({
        "geoid": acs["geoid"],
        "year": acs["year"],
    })

    # -------------------------
    # Population / age
    # -------------------------
    out["pop_total"] = acs["B01001_001E"]

    out["pop_under18"] = (
        acs["B01001_003E"] + acs["B01001_004E"] + acs["B01001_005E"] + acs["B01001_006E"] +
        acs["B01001_027E"] + acs["B01001_028E"] + acs["B01001_029E"] + acs["B01001_030E"]
    )
    out["share_under18"] = safe_div(out["pop_under18"], out["pop_total"])

    out["pop_65plus"] = (
        acs["B01001_020E"] + acs["B01001_021E"] + acs["B01001_022E"] + acs["B01001_023E"] + acs["B01001_024E"] + acs["B01001_025E"] +
        acs["B01001_044E"] + acs["B01001_045E"] + acs["B01001_046E"] + acs["B01001_047E"] + acs["B01001_048E"] + acs["B01001_049E"]
    )
    out["share_65plus"] = safe_div(out["pop_65plus"], out["pop_total"])

    # -------------------------
    # Race shares
    # -------------------------
    out["share_white"] = safe_div(acs["B02001_002E"], acs["B02001_001E"])
    out["share_black"] = safe_div(acs["B02001_003E"], acs["B02001_001E"])
    out["share_asian"] = safe_div(acs["B02001_005E"], acs["B02001_001E"])
    out["share_two_plus"] = safe_div(acs["B02001_008E"], acs["B02001_001E"])

    # -------------------------
    # Education
    # -------------------------
    ba_plus = acs["B15003_022E"] + acs["B15003_023E"] + acs["B15003_024E"] + acs["B15003_025E"]
    out["ba_plus_share_25p"] = safe_div(ba_plus, acs["B15003_001E"])

    # -------------------------
    # Employment / economy
    # -------------------------
    out["lfpr_16p"] = safe_div(acs["B23025_002E"], acs["B23025_001E"])
    out["unemp_rate"] = safe_div(acs["B23025_005E"], acs["B23025_003E"])

    out["median_household_income"] = acs["B19013_001E"]
    out["per_capita_income"] = acs["B19301_001E"]
    out["gini"] = acs["B19083_001E"]
    out["poverty_rate"] = safe_div(acs["B17001_002E"], acs["B17001_001E"])

    # -------------------------
    # Commute / internet
    # -------------------------
    commute_under35 = (
        acs["B08303_002E"] + acs["B08303_003E"] + acs["B08303_004E"] + acs["B08303_005E"] +
        acs["B08303_006E"] + acs["B08303_007E"] + acs["B08303_008E"]
    )
    out["commute_under35_share"] = safe_div(commute_under35, acs["B08303_001E"])
    out["internet_sub_share"] = safe_div(acs["B28002_002E"], acs["B28002_001E"])

    # -------------------------
    # Housing / affordability
    # -------------------------
    out["housing_units_total"] = acs["B25002_001E"]
    out["occupied_units"] = acs["B25002_002E"]
    out["vacant_units"] = acs["B25002_003E"]
    out["occupied_share"] = safe_div(acs["B25002_002E"], acs["B25002_001E"])
    out["vacancy_rate"] = safe_div(acs["B25002_003E"], acs["B25002_001E"])

    out["owner_share"] = safe_div(acs["B25003_002E"], acs["B25003_001E"])
    out["renter_share"] = safe_div(acs["B25003_003E"], acs["B25003_001E"])

    out["median_gross_rent"] = acs["B25064_001E"]
    out["median_home_value"] = acs["B25077_001E"]

    rent_affordable = (
        acs["B25070_002E"] + acs["B25070_003E"] + acs["B25070_004E"] +
        acs["B25070_005E"] + acs["B25070_006E"]
    )

    owner_affordable = (
        acs["B25091_003E"] + acs["B25091_004E"] + acs["B25091_005E"] + acs["B25091_006E"] + acs["B25091_007E"] +
        acs["B25091_014E"] + acs["B25091_015E"] + acs["B25091_016E"] + acs["B25091_017E"] + acs["B25091_018E"]
    )

    denom_housing_cost = acs["B25070_001E"] + acs["B25091_001E"]
    out["affordable_housing_share"] = safe_div(rent_affordable + owner_affordable, denom_housing_cost)

    # -------------------------
    # Community / health
    # -------------------------
    # Early education proxy: enrolled 3-4 year olds over under-5 population
    enrolled_3_4 = acs["B14003_004E"] + acs["B14003_013E"] + acs["B14003_032E"] + acs["B14003_041E"]
    pop_under5 = acs["B01001_003E"] + acs["B01001_027E"]
    out["early_ed_enroll_share"] = safe_div(enrolled_3_4, pop_under5)

    uninsured = (
        acs["B27001_005E"] + acs["B27001_008E"] + acs["B27001_011E"] + acs["B27001_014E"] + acs["B27001_017E"] +
        acs["B27001_020E"] + acs["B27001_023E"] + acs["B27001_026E"] + acs["B27001_029E"] +
        acs["B27001_033E"] + acs["B27001_036E"] + acs["B27001_039E"] + acs["B27001_042E"] + acs["B27001_045E"] +
        acs["B27001_048E"] + acs["B27001_051E"] + acs["B27001_054E"] + acs["B27001_057E"]
    )
    out["insured_share"] = 1.0 - safe_div(uninsured, acs["B27001_001E"])

    # -------------------------
    # Growth features
    # -------------------------
    out = out.sort_values(["geoid", "year"]).reset_index(drop=True)

    growth_cols = [
        "median_household_income",
        "per_capita_income",
        "occupied_units",
        "median_home_value",
        "median_gross_rent",
    ]

    for col in growth_cols:
        out[f"{col}_growth"] = out.groupby("geoid")[col].pct_change(fill_method=None)

    return out

### Cell 7 — Merge helpers

In [43]:
def prepare_acs_raw_for_merge(acs: pd.DataFrame) -> pd.DataFrame:
    acs_raw = acs.copy()

    acs_raw["state"] = acs_raw["state"].astype(str).str.zfill(2)
    acs_raw["county"] = acs_raw["county"].astype(str).str.zfill(3)
    acs_raw["tract"] = acs_raw["tract"].astype(str).str.zfill(6)
    acs_raw["geoid"] = acs_raw["state"] + acs_raw["county"] + acs_raw["tract"]

    # one row per tract-year expected
    acs_raw = acs_raw.drop_duplicates(subset=["geoid", "year"]).copy()

    return acs_raw


def merge_igs_acs_raw(igs: pd.DataFrame, acs_raw: pd.DataFrame) -> pd.DataFrame:
    merged = igs.merge(acs_raw, on=["geoid", "year"], how="left", validate="m:1")
    return merged


def merge_igs_acs_full(
    igs: pd.DataFrame,
    acs_raw: pd.DataFrame,
    feats: pd.DataFrame,
    
) -> pd.DataFrame:
    merged = (
        igs
        .merge(acs_raw, on=["geoid", "year"], how="left", validate="m:1")
        .merge(feats, on=["geoid", "year"], how="left", validate="m:1")
    )
    return merged

In [44]:
# for year in range(2017, 2025):
#     p = RAW_DIR / f"acs5_tract_{year}.parquet"
#     if p.exists():
#         p.unlink()
#         print("deleted", p.name)

### Cell 8 — Run ACS pull, feature engineering, and merge

In [45]:
igs_years = sorted(igs["year"].dropna().unique().tolist())
acs_years = resolve_acs_years(igs_years, max_available_year=ACS_MAX_AVAILABLE_YEAR)
state_fips_filter = infer_state_fips_from_tracts(TARGET_TRACTS)

print("IGS years in file:", igs_years)
print("ACS years to pull:", acs_years)
print("State filter from target tracts:", state_fips_filter)

acs = download_acs_all(
    years=acs_years,
    api_key=CENSUS_API_KEY,
    state_fips_filter=state_fips_filter,
    max_vars_per_call=45,
    sleep_seconds=0.25,
)

acs_raw = prepare_acs_raw_for_merge(acs)
feats = make_features(acs)
feats = filter_tracts(feats, TARGET_TRACTS)

# keep only IGS years that exist in ACS-derived tables
valid_years = sorted(feats["year"].dropna().unique())
model_igs = igs[igs["year"].isin(valid_years)].copy()

igs_x_acs_raw = merge_igs_acs_raw(model_igs, acs_raw)
model_df_full = merge_igs_acs_full(model_igs, acs_raw, feats)

if SAVE_INTERMEDIATE:
    safe_save_parquet(acs, OUT_DIR / "acs_raw.parquet")
    safe_save_parquet(acs_raw, OUT_DIR / "acs_raw_with_geoid.parquet")
    safe_save_parquet(feats, OUT_DIR / "acs_features.parquet")
    safe_save_parquet(igs_x_acs_raw, OUT_DIR / "igs_x_acs_raw.parquet")
    safe_save_parquet(model_df_full, OUT_DIR / "igs_x_acs_full.parquet")

print("ACS raw shape:", acs.shape)
print("ACS raw+geoid shape:", acs_raw.shape)
print("ACS features shape:", feats.shape)
print("IGS x ACS raw shape:", igs_x_acs_raw.shape)
print("Full merged model_df_full shape:", model_df_full.shape)

IGS years in file: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
ACS years to pull: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
State filter from target tracts: None
[cache] acs5_tract_2017.parquet
[cache] acs5_tract_2018.parquet
[cache] acs5_tract_2019.parquet
[cache] acs5_tract_2020.parquet
[cache] acs5_tract_2021.parquet
[cache] acs5_tract_2022.parquet
[cache] acs5_tract_2023.parquet
[cache] acs5_tract_2024.parquet
ACS raw shape: (648952, 103)
ACS raw+geoid shape: (648952, 104)
ACS features shape: (648952, 37)
IGS x ACS raw shape: (680256, 189)
Full merged model_df_full shape: (680256, 224)


### validation cell right after Cell 8

In [46]:
# choose the final EDA panel
eda_df = model_df_full.copy()

print("EDA shape:", eda_df.shape)
print("EDA years:", sorted(eda_df["year"].dropna().unique().tolist()))
print("EDA unique tracts:", eda_df["geoid"].nunique())

# required ACS raw columns
required_acs_raw = ["NAME", "state", "county", "tract"] + ACS_VARS
missing_acs_raw = [c for c in required_acs_raw if c not in eda_df.columns]

# required engineered columns
required_feats = [c for c in feats.columns if c not in ["geoid", "year"]]
missing_feats = [c for c in required_feats if c not in eda_df.columns]

# all original IGS columns
missing_igs = [c for c in igs.columns if c not in eda_df.columns]

print("Missing raw ACS columns:", missing_acs_raw[:20], "count =", len(missing_acs_raw))
print("Missing engineered feature columns:", missing_feats[:20], "count =", len(missing_feats))
print("Missing IGS columns:", missing_igs[:20], "count =", len(missing_igs))

print("Duplicate geoid-year rows:", eda_df.duplicated(subset=["geoid", "year"]).sum())

# quick sample
cols_to_show = [
    "geoid", "year", "igs_total", "igs_place", "igs_economy", "igs_community",
    "NAME", "state", "county", "tract",
    "B19013_001E", "B19301_001E", "B25077_001E",
    "median_household_income", "per_capita_income", "median_home_value"
]
print(eda_df[cols_to_show].head())

EDA shape: (680256, 224)
EDA years: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
EDA unique tracts: 85032
Missing raw ACS columns: [] count = 0
Missing engineered feature columns: [] count = 0
Missing IGS columns: [] count = 0
Duplicate geoid-year rows: 0
         geoid  year  igs_total  igs_place  igs_economy  igs_community                                       NAME state county   tract  B19013_001E  B19301_001E  B25077_001E  \
0  01001020100  2017       47.0       53.0         34.0           54.0  Census Tract 201, Autauga County, Alabama    01    001  020100      67826.0      33018.0     152500.0   
1  01001020100  2018       52.0       48.0         48.0           62.0  Census Tract 201, Autauga County, Alabama    01    001  020100      58625.0      31580.0     133300.0   
2  01001020100  2019       46.0       40.0         40.0           56.0  Census Tract 201, Autauga County, Alabama    01    001  020100      60208.0      31225.0     136100.0   
3  01001020100  2020       46.0 

### Cell 9 — Final checks before EDA

In [47]:
print("Merged years:", sorted(model_df_full["year"].dropna().unique().tolist()))
print()

print(model_df_full[[
    "geoid", "year", "igs_total", "igs_place", "igs_economy", "igs_community",
    "affordable_housing_share", "internet_sub_share", "insured_share",
    "median_household_income", "poverty_rate", "vacancy_rate"
]].head())

print()
print("Top missingness in merged data:")
print(model_df_full.isna().mean().sort_values(ascending=False).head(30))

print()
print("Core numeric summary:")
core_cols = [
    "igs_total",
    "igs_place",
    "igs_economy",
    "igs_community",
    "affordable_housing_share",
    "internet_sub_share",
    "insured_share",
    "median_household_income",
    "per_capita_income",
    "poverty_rate",
    "vacancy_rate",
    "unemp_rate",
]
print(model_df_full[core_cols].describe())

Merged years: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

         geoid  year  igs_total  igs_place  igs_economy  igs_community  affordable_housing_share  internet_sub_share  insured_share  median_household_income  poverty_rate  vacancy_rate
0  01001020100  2017       47.0       53.0         34.0           54.0                  0.737401            0.762599       0.909485                  67826.0      0.106775      0.014379
1  01001020100  2018       52.0       48.0         48.0           62.0                  0.704575            0.717647       0.907436                  58625.0      0.113365      0.017972
2  01001020100  2019       46.0       40.0         40.0           56.0                  0.715092            0.740480       0.902659                  60208.0      0.166583      0.022069
3  01001020100  2020       46.0       34.0         48.0           56.0                  0.738817            0.810967       0.903658                  60388.0      0.136528      0.023944
4  01001020

###  Cell 10 — Optional: save an EDA subset for only chosen tracts

In [48]:
# Cell 10 — final EDA panel cleanup + save

eda_df = model_df_full.copy()   # use model_df_full, not model_df

# drop junk unnamed / NaN column names from the original IGS export
junk_cols = [c for c in eda_df.columns if pd.isna(c) or str(c).startswith("Unnamed")]
print("Dropping junk columns:", junk_cols)

eda_df = eda_df.drop(columns=junk_cols, errors="ignore")

# optional: reorder key columns to the front
front_cols = [
    c for c in [
        "geoid", "year", "Census Tract FIPS code", "County", "State",
        "igs_total", "igs_place", "igs_economy", "igs_community"
    ] if c in eda_df.columns
]
other_cols = [c for c in eda_df.columns if c not in front_cols]
eda_df = eda_df[front_cols + other_cols]

print("EDA panel shape:", eda_df.shape)

if SAVE_INTERMEDIATE:
    safe_save_parquet(eda_df, OUT_DIR / "eda_panel_clean.parquet")

eda_df.head()

Dropping junk columns: []
EDA panel shape: (680256, 224)


,geoid,year,Census Tract FIPS code,County,State,igs_total,igs_place,igs_economy,igs_community,nan,Is an Opportunity Zone,Year,Inclusive Growth Score,Growth,Inclusion,Place,Place Growth,Place Inclusion,Net Occupancy Score,"Net Occupancy Base, %","Net Occupancy Tract, %",Residential Real Estate Value Score,"Residential Real Estate Value Base, %","Residential Real Estate Value Tract, %",Acres of Park Land Score,"Acres of Park Land Base, %","Acres of Park Land Tract, %",Affordable Housing Score,"Affordable Housing Base, %","Affordable Housing Tract, %",Internet Access Score,"Internet Access Base, %","Internet Access Tract, %",Travel Time to Work Score,"Travel Time to Work Base, %","Travel Time to Work Tract, %",Economy,Economy Growth,Economy Inclusion,New Businesses Score,"New Businesses Base, %","New Businesses Tract, %",Spend Growth Score,"Spend Growth Base, %","Spend Growth Tract, %",Small Business Loans Score,"Small Business Loans Base, %","Small Business Loans Tract, %",Minority/Women Owned Businesses Score,"Minority/Women Owned Businesses Base, %","Minority/Women Owned Businesses Tract, %",Labor Market Engagement Index Score,Labor Market Engagement Index Base,Labor Market Engagement Index Tract,Commercial Diversity Score,"Commercial Diversity Base, %","Commercial Diversity Tract, %",Community,Community Growth,Community Inclusion,Personal Income Score,"Personal Income Base, %","Personal Income Tract, %",Spending per Capita Score,"Spending per Capita Base, %","Spending per Capita Tract, %",Female Above Poverty Score,"Female Above Poverty Base, %","Female Above Poverty Tract, %",Gini Coefficient Score,Gini Coefficient Base,Gini Coefficient Tract,Early Education Enrollment Score,"Early Education Enrollment Base, %","Early Education Enrollment Tract, %",Health Insurance Coverage Score,"Health Insurance Coverage Base, %","Health Insurance Coverage Tract, %",igs_growth,igs_inclusion,igs_place_growth,igs_place_inclusion,igs_economy_growth,igs_economy_inclusion,igs_community_growth,igs_community_inclusion,is_opp_zone,NAME,B01001_001E,B01001_003E,B01001_004E,B01001_005E,B01001_006E,B01001_020E,B01001_021E,B01001_022E,B01001_023E,B01001_024E,B01001_025E,B01001_027E,...,B14003_032E,B14003_041E,B15003_001E,B15003_022E,B15003_023E,B15003_024E,B15003_025E,B17001_001E,B17001_002E,state,county,tract,B19013_001E,B19083_001E,B19301_001E,B23025_001E,B23025_002E,B23025_003E,B23025_005E,B25002_001E,B25002_002E,B25002_003E,B25003_001E,B25003_002E,B25003_003E,B25064_001E,B25070_001E,B25070_002E,B25070_003E,B25070_004E,B25070_005E,B25070_006E,B25077_001E,B25091_001E,B25091_003E,B25091_004E,B25091_005E,B25091_006E,B25091_007E,B25091_014E,B25091_015E,B25091_016E,B25091_017E,B25091_018E,B27001_001E,B27001_005E,B27001_008E,B27001_011E,B27001_014E,B27001_017E,B27001_020E,B27001_023E,B27001_026E,B27001_029E,B27001_033E,B27001_036E,B27001_039E,B27001_042E,B27001_045E,B27001_048E,B27001_051E,B27001_054E,B27001_057E,B28002_001E,B28002_002E,pop_total,pop_under18,share_under18,pop_65plus,share_65plus,share_white,share_black,share_asian,share_two_plus,ba_plus_share_25p,lfpr_16p,unemp_rate,median_household_income,per_capita_income,gini,poverty_rate,commute_under35_share,internet_sub_share,housing_units_total,occupied_units,vacant_units,occupied_share,vacancy_rate,owner_share,renter_share,median_gross_rent,median_home_value,affordable_housing_share,early_ed_enroll_share,insured_share,median_household_income_growth,per_capita_income_growth,occupied_units_growth,median_home_value_growth,median_gross_rent_growth
0,01001020100,2017,01001020100,Autauga County,Alabama,47.0,53.0,34.0,54.0,8100.0,NaN,2017,47.0,45.0,49.0,53.0,62.0,44.0,62.0,1.5,11.4,63.0,-1.2,15.7,29.0,3.1,1.4,98.0,76.5,90.2,43.0,78.6,76.3,5.0,74.1,46.1,34.0,30.0,38.0,67.0,4.1,57.1,10.0,NaN,NaN,14.0,4.5,-15.8,0.0,0.0,0.0,67.0,48.0,65.0,8.0,20.3,10.5,54.0,42.0,66.0,33.0,3.0,-4.2,52.0,NaN,NaN,78.0,83.7,91.6,44.0,41.4,42.3,96.0,23.1,53.2,44.0,90.0,88.9,45.0,49.0,62.0,44.0,30.0,38.0,42.0,66.0,NaN,"Census Trac

### Cell 11 — validation checks


In [49]:
# Cell 11 — validation checks

print("Duplicate geoid-year rows:", eda_df.duplicated(subset=["geoid", "year"]).sum())

required_acs_raw = ["NAME", "state", "county", "tract"] + ACS_VARS
missing_acs_raw = [c for c in required_acs_raw if c not in eda_df.columns]

required_feats = [c for c in feats.columns if c not in ["geoid", "year"]]
missing_feats = [c for c in required_feats if c not in eda_df.columns]

missing_igs = [c for c in igs.columns if c not in eda_df.columns]

print("Missing raw ACS columns:", len(missing_acs_raw))
print("Missing engineered ACS columns:", len(missing_feats))
print("Missing IGS columns:", len(missing_igs))

Duplicate geoid-year rows: 0
Missing raw ACS columns: 0
Missing engineered ACS columns: 0
Missing IGS columns: 0


### Cell 12 — shortlist settings + required columns

In [50]:
# Cell 12 — shortlist settings + required columns

SHORTLIST_YEAR = 2024
LOW_IGS_THRESHOLD = 45
PERSIST_YEARS = [2022, 2023, 2024]

TOP_N_NATIONAL = 150
TOP_N_PER_STATE = 3
MIN_POP_FLAG = 1500  # flag only, do not drop automatically

shortlist_required_cols = [
    "geoid", "year",
    "NAME", "state", "county", "tract",
    "County", "State",
    "igs_total", "igs_place", "igs_economy", "igs_community",
    "pop_total", "share_under18", "share_65plus",
    "lfpr_16p", "unemp_rate",
    "median_household_income", "poverty_rate",
    "internet_sub_share", "vacancy_rate",
    "affordable_housing_share", "insured_share",
]

missing_shortlist_cols = [c for c in shortlist_required_cols if c not in eda_df.columns]
print("Missing shortlist columns:", missing_shortlist_cols)

if missing_shortlist_cols:
    raise ValueError(f"Missing required shortlist columns: {missing_shortlist_cols}")

print("Shortlist year:", SHORTLIST_YEAR)
print("Low-IGS threshold:", LOW_IGS_THRESHOLD)
print("Persistence years:", PERSIST_YEARS)

Missing shortlist columns: []
Shortlist year: 2024
Low-IGS threshold: 45
Persistence years: [2022, 2023, 2024]


### Cell 13 — scoring helpers

In [51]:
# Cell 13 — scoring helpers

def weighted_percentile_points(series: pd.Series, weight: float, higher_is_worse: bool) -> pd.Series:
    """
    Convert a column into weighted percentile-rank points.
    - higher_is_worse=True  -> larger values get more points
    - higher_is_worse=False -> smaller values get more points
    """
    ranked = series.rank(
        method="average",
        pct=True,
        ascending=higher_is_worse
    )
    return ranked * weight

def add_score(df: pd.DataFrame, col: str, weight: float, higher_is_worse: bool, out_col: str) -> pd.DataFrame:
    df[out_col] = weighted_percentile_points(df[col], weight=weight, higher_is_worse=higher_is_worse)
    return df

### Cell 14 — build 2024 base + persistence table

In [52]:
# Cell 14 — build 2024 base + persistence table

base_2024 = eda_df.loc[eda_df["year"] == SHORTLIST_YEAR].copy()

# display fields: prefer human-readable IGS fields, then fall back to ACS fields
base_2024["display_state"] = base_2024["State"].fillna(base_2024["state"])
base_2024["display_county"] = base_2024["County"].fillna(base_2024["county"])
base_2024["display_name"] = base_2024["NAME"]

persistence = (
    eda_df.loc[eda_df["year"].isin(PERSIST_YEARS), ["geoid", "year", "igs_total"]]
    .dropna(subset=["igs_total"])
    .assign(igs_below_45=lambda d: d["igs_total"] < LOW_IGS_THRESHOLD)
    .groupby("geoid", as_index=False)
    .agg(
        n_persist_years=("igs_below_45", "sum"),
        mean_igs_persist_window=("igs_total", "mean"),
        min_igs_persist_window=("igs_total", "min"),
    )
)

persistence["persistence_score"] = persistence["n_persist_years"].clip(lower=0, upper=3).astype(float)

shortlist_base = (
    base_2024
    .merge(persistence, on="geoid", how="left")
    .copy()
)

shortlist_base["n_persist_years"] = shortlist_base["n_persist_years"].fillna(0).astype(int)
shortlist_base["persistence_score"] = shortlist_base["persistence_score"].fillna(0.0)

# main eligibility rule
shortlist_base = shortlist_base.loc[shortlist_base["igs_total"] < LOW_IGS_THRESHOLD].copy()

# tiny tract flag only
shortlist_base["tiny_tract_flag"] = shortlist_base["pop_total"] < MIN_POP_FLAG

print("2024 tracts total:", len(base_2024))
print("Eligible low-IGS tracts:", len(shortlist_base))
print("Unique states in shortlist:", shortlist_base["display_state"].nunique())
print()

print(shortlist_base[
    [
        "geoid", "display_state", "display_county", "display_name",
        "igs_total", "igs_place", "igs_economy", "igs_community",
        "n_persist_years", "persistence_score"
    ]
].head())

2024 tracts total: 85032
Eligible low-IGS tracts: 24329
Unique states in shortlist: 52

          geoid display_state  display_county                                  display_name  igs_total  igs_place  igs_economy  igs_community  n_persist_years  persistence_score
0   01001020100       Alabama  Autauga County     Census Tract 201; Autauga County; Alabama       40.0       43.0         38.0           40.0                3                3.0
10  01001020803       Alabama  Autauga County  Census Tract 208.03; Autauga County; Alabama       42.0       30.0         56.0           39.0                1                1.0
11  01001020804       Alabama  Autauga County  Census Tract 208.04; Autauga County; Alabama       42.0       30.0         56.0           39.0                1                1.0
12  01001020805       Alabama  Autauga County  Census Tract 208.05; Autauga County; Alabama       42.0       30.0         56.0           39.0                1                1.0
13  01001020901       

### Cell 15 — apply the exact shortlist scorecard

In [53]:
# Cell 15 — apply the exact shortlist scorecard

score_input_cols = [
    "igs_total", "igs_place", "igs_economy", "igs_community",
    "poverty_rate", "insured_share", "median_household_income",
    "unemp_rate", "affordable_housing_share", "internet_sub_share",
    "share_65plus", "share_under18", "lfpr_16p", "vacancy_rate"
]

tract_shortlist_scored = shortlist_base.dropna(subset=score_input_cols).copy()

# --------------------------
# A. IGS distress (45 total)
# --------------------------
tract_shortlist_scored = add_score(tract_shortlist_scored, "igs_total", 20, higher_is_worse=False, out_col="igs_total_pts")
tract_shortlist_scored = add_score(tract_shortlist_scored, "igs_economy", 10, higher_is_worse=False, out_col="igs_economy_pts")
tract_shortlist_scored = add_score(tract_shortlist_scored, "igs_place", 8, higher_is_worse=False, out_col="igs_place_pts")
tract_shortlist_scored = add_score(tract_shortlist_scored, "igs_community", 4, higher_is_worse=False, out_col="igs_community_pts")

tract_shortlist_scored["igs_distress_score"] = (
    tract_shortlist_scored["igs_total_pts"] +
    tract_shortlist_scored["igs_economy_pts"] +
    tract_shortlist_scored["igs_place_pts"] +
    tract_shortlist_scored["igs_community_pts"] +
    tract_shortlist_scored["persistence_score"]
)

# ----------------------------------------
# B. Healthcare proxy pressure (35 total)
# ----------------------------------------
tract_shortlist_scored = add_score(tract_shortlist_scored, "poverty_rate", 10, higher_is_worse=True, out_col="poverty_rate_pts")
tract_shortlist_scored = add_score(tract_shortlist_scored, "insured_share", 9, higher_is_worse=False, out_col="insured_share_pts")
tract_shortlist_scored = add_score(tract_shortlist_scored, "median_household_income", 7, higher_is_worse=False, out_col="median_household_income_pts")
tract_shortlist_scored = add_score(tract_shortlist_scored, "unemp_rate", 5, higher_is_worse=True, out_col="unemp_rate_pts")
tract_shortlist_scored = add_score(tract_shortlist_scored, "affordable_housing_share", 2, higher_is_worse=False, out_col="affordable_housing_share_pts")
tract_shortlist_scored = add_score(tract_shortlist_scored, "internet_sub_share", 2, higher_is_worse=False, out_col="internet_sub_share_pts")

tract_shortlist_scored["healthcare_proxy_score"] = (
    tract_shortlist_scored["poverty_rate_pts"] +
    tract_shortlist_scored["insured_share_pts"] +
    tract_shortlist_scored["median_household_income_pts"] +
    tract_shortlist_scored["unemp_rate_pts"] +
    tract_shortlist_scored["affordable_housing_share_pts"] +
    tract_shortlist_scored["internet_sub_share_pts"]
)

# ------------------------------------
# C. Vulnerability / need (20 total)
# ------------------------------------
tract_shortlist_scored = add_score(tract_shortlist_scored, "share_65plus", 6, higher_is_worse=True, out_col="share_65plus_pts")
tract_shortlist_scored = add_score(tract_shortlist_scored, "share_under18", 4, higher_is_worse=True, out_col="share_under18_pts")
tract_shortlist_scored = add_score(tract_shortlist_scored, "lfpr_16p", 5, higher_is_worse=False, out_col="lfpr_16p_pts")
tract_shortlist_scored = add_score(tract_shortlist_scored, "vacancy_rate", 5, higher_is_worse=True, out_col="vacancy_rate_pts")

tract_shortlist_scored["vulnerability_score"] = (
    tract_shortlist_scored["share_65plus_pts"] +
    tract_shortlist_scored["share_under18_pts"] +
    tract_shortlist_scored["lfpr_16p_pts"] +
    tract_shortlist_scored["vacancy_rate_pts"]
)

# final total
tract_shortlist_scored["total_shortlist_score"] = (
    tract_shortlist_scored["igs_distress_score"] +
    tract_shortlist_scored["healthcare_proxy_score"] +
    tract_shortlist_scored["vulnerability_score"]
)

# keep a clean ranking
tract_shortlist_scored = tract_shortlist_scored.sort_values(
    ["total_shortlist_score", "igs_total"],
    ascending=[False, True]
).reset_index(drop=True)

print("Scored shortlist rows:", len(tract_shortlist_scored))
print()

print(tract_shortlist_scored[
    [
        "geoid", "display_state", "display_county", "display_name",
        "igs_total", "igs_place", "igs_economy", "igs_community",
        "igs_distress_score", "healthcare_proxy_score", "vulnerability_score",
        "total_shortlist_score"
    ]
].head(10))

Scored shortlist rows: 23873

         geoid display_state      display_county                                      display_name  igs_total  igs_place  igs_economy  igs_community  igs_distress_score  healthcare_proxy_score  \
0  47157011500     Tennessee       Shelby County        Census Tract 115; Shelby County; Tennessee       25.0       25.0         28.0           23.0           42.363716               32.687010   
1  72127003300   Puerto Rico  San Juan Municipio  Census Tract 33; San Juan Municipio; Puerto Rico       22.0       32.0         21.0           14.0           42.547313               32.940812   
2  47157005900     Tennessee       Shelby County         Census Tract 59; Shelby County; Tennessee       22.0       28.0         30.0            8.0           41.972731               32.325514   
3  72127003600   Puerto Rico  San Juan Municipio  Census Tract 36; San Juan Municipio; Puerto Rico       25.1       32.8         18.6           23.7           42.073933               33.

### Cell 15.5 — shortlist audit

In [54]:
# Cell 15.5 — shortlist audit

print("=== SHORTLIST AUDIT ===")
print()

# 1) Row-count audit
eligible_n = len(shortlist_base)
scored_n = len(tract_shortlist_scored)
dropped_n = eligible_n - scored_n
dropped_pct = (dropped_n / eligible_n * 100) if eligible_n else 0

print("Row-count audit")
print(f"Eligible low-IGS tracts in shortlist_base: {eligible_n:,}")
print(f"Scored tracts in tract_shortlist_scored: {scored_n:,}")
print(f"Dropped before scoring: {dropped_n:,} ({dropped_pct:.2f}%)")
print()

# 2) Missingness audit for the scoring inputs
print("Missingness audit on score_input_cols within shortlist_base")
missing_summary = pd.DataFrame({
    "missing_count": shortlist_base[score_input_cols].isna().sum(),
    "missing_pct": shortlist_base[score_input_cols].isna().mean().mul(100)
}).sort_values(["missing_count", "missing_pct"], ascending=[False, False])

print(missing_summary)
print()

# 3) Which columns are causing drops specifically?
dropped_rows = shortlist_base[shortlist_base[score_input_cols].isna().any(axis=1)].copy()

print("Rows dropped because at least one score input is missing:", len(dropped_rows))
print()

if len(dropped_rows) > 0:
    dropped_missing_summary = pd.DataFrame({
        "missing_count_among_dropped": dropped_rows[score_input_cols].isna().sum(),
        "missing_pct_among_dropped": dropped_rows[score_input_cols].isna().mean().mul(100)
    }).sort_values(["missing_count_among_dropped", "missing_pct_among_dropped"], ascending=[False, False])

    print("Missingness among dropped rows only")
    print(dropped_missing_summary)
    print()

    print("Sample dropped rows")
    display_cols = [
        "geoid", "display_state", "display_county", "display_name",
        "igs_total", "igs_place", "igs_economy", "igs_community"
    ] + score_input_cols

    print(dropped_rows[display_cols].head(10))
    print()

# 4) Score-construction audit
print("Score construction audit")

tract_shortlist_scored["igs_distress_score_check"] = (
    tract_shortlist_scored["igs_total_pts"] +
    tract_shortlist_scored["igs_economy_pts"] +
    tract_shortlist_scored["igs_place_pts"] +
    tract_shortlist_scored["igs_community_pts"] +
    tract_shortlist_scored["persistence_score"]
)

tract_shortlist_scored["healthcare_proxy_score_check"] = (
    tract_shortlist_scored["poverty_rate_pts"] +
    tract_shortlist_scored["insured_share_pts"] +
    tract_shortlist_scored["median_household_income_pts"] +
    tract_shortlist_scored["unemp_rate_pts"] +
    tract_shortlist_scored["affordable_housing_share_pts"] +
    tract_shortlist_scored["internet_sub_share_pts"]
)

tract_shortlist_scored["vulnerability_score_check"] = (
    tract_shortlist_scored["share_65plus_pts"] +
    tract_shortlist_scored["share_under18_pts"] +
    tract_shortlist_scored["lfpr_16p_pts"] +
    tract_shortlist_scored["vacancy_rate_pts"]
)

tract_shortlist_scored["total_shortlist_score_check"] = (
    tract_shortlist_scored["igs_distress_score_check"] +
    tract_shortlist_scored["healthcare_proxy_score_check"] +
    tract_shortlist_scored["vulnerability_score_check"]
)

score_audit = pd.DataFrame({
    "igs_distress_max_abs_diff": [
        (tract_shortlist_scored["igs_distress_score"] - tract_shortlist_scored["igs_distress_score_check"]).abs().max()
    ],
    "healthcare_proxy_max_abs_diff": [
        (tract_shortlist_scored["healthcare_proxy_score"] - tract_shortlist_scored["healthcare_proxy_score_check"]).abs().max()
    ],
    "vulnerability_max_abs_diff": [
        (tract_shortlist_scored["vulnerability_score"] - tract_shortlist_scored["vulnerability_score_check"]).abs().max()
    ],
    "total_score_max_abs_diff": [
        (tract_shortlist_scored["total_shortlist_score"] - tract_shortlist_scored["total_shortlist_score_check"]).abs().max()
    ],
})

print(score_audit)
print()

# 5) Score-range audit
print("Score range audit")
range_audit = pd.DataFrame({
    "min": tract_shortlist_scored[
        ["igs_distress_score", "healthcare_proxy_score", "vulnerability_score", "total_shortlist_score"]
    ].min(),
    "max": tract_shortlist_scored[
        ["igs_distress_score", "healthcare_proxy_score", "vulnerability_score", "total_shortlist_score"]
    ].max(),
    "mean": tract_shortlist_scored[
        ["igs_distress_score", "healthcare_proxy_score", "vulnerability_score", "total_shortlist_score"]
    ].mean(),
})
print(range_audit)
print()

# 6) Persistence audit
print("Persistence audit")
print(tract_shortlist_scored["n_persist_years"].value_counts(dropna=False).sort_index())
print()
print("Persistence score min/max:",
      tract_shortlist_scored["persistence_score"].min(),
      tract_shortlist_scored["persistence_score"].max())
print()

# 7) Tiny-tract audit among top-ranked rows
print("Tiny-tract audit in top 25 scored rows")
print(
    tract_shortlist_scored.head(25)[
        ["geoid", "display_state", "display_county", "igs_total", "total_shortlist_score", "tiny_tract_flag"]
    ]
)
print()

print("=== END SHORTLIST AUDIT ===")

=== SHORTLIST AUDIT ===

Row-count audit
Eligible low-IGS tracts in shortlist_base: 24,329
Scored tracts in tract_shortlist_scored: 23,873
Dropped before scoring: 456 (1.87%)

Missingness audit on score_input_cols within shortlist_base
                          missing_count  missing_pct
median_household_income             456     1.874306
affordable_housing_share            227     0.933043
internet_sub_share                  227     0.933043
vacancy_rate                        220     0.904271
poverty_rate                        213     0.875498
insured_share                       197     0.809733
unemp_rate                          192     0.789182
share_65plus                        143     0.587776
share_under18                       143     0.587776
lfpr_16p                            143     0.587776
igs_total                             0     0.000000
igs_place                             0     0.000000
igs_economy                           0     0.000000
igs_community         

### Cell 16 — build the main shortlist outputs

In [55]:
# Cell 16 — build the main shortlist outputs

shortlist_output_cols = [
    "geoid", "year",
    "display_state", "display_county", "display_name",
    "State", "County", "state", "county", "tract",
    "igs_total", "igs_place", "igs_economy", "igs_community",
    "n_persist_years", "mean_igs_persist_window", "min_igs_persist_window",
    "pop_total", "share_under18", "share_65plus",
    "lfpr_16p", "unemp_rate", "median_household_income", "poverty_rate",
    "internet_sub_share", "vacancy_rate", "affordable_housing_share", "insured_share",
    "tiny_tract_flag",
    "igs_distress_score", "healthcare_proxy_score", "vulnerability_score",
    "total_shortlist_score"
]

tract_shortlist_scored = tract_shortlist_scored[shortlist_output_cols].copy()

top_national_candidates = tract_shortlist_scored.head(TOP_N_NATIONAL).copy()

state_balanced_candidates = (
    tract_shortlist_scored
    .sort_values(["display_state", "total_shortlist_score"], ascending=[True, False])
    .groupby("display_state", group_keys=False)
    .head(TOP_N_PER_STATE)
    .reset_index(drop=True)
)

print("Top national candidates:", top_national_candidates.shape)
print("State-balanced candidates:", state_balanced_candidates.shape)
print()

print(top_national_candidates[
    ["geoid", "display_state", "display_county", "igs_total", "total_shortlist_score"]
].head(20))

print()
print(state_balanced_candidates[
    ["display_state", "geoid", "display_county", "igs_total", "total_shortlist_score"]
].head(30))

Top national candidates: (150, 33)
State-balanced candidates: (153, 33)

          geoid  display_state      display_county  igs_total  total_shortlist_score
0   47157011500      Tennessee       Shelby County       25.0              91.759100
1   72127003300    Puerto Rico  San Juan Municipio       22.0              91.089515
2   47157005900      Tennessee       Shelby County       22.0              90.629875
3   72127003600    Puerto Rico  San Juan Municipio       25.1              90.228291
4   04017940012        Arizona       Navajo County       24.0              90.182591
5   47157011200      Tennessee       Shelby County       22.0              89.833159
6   46071941200   South Dakota      Jackson County       18.0              89.370272
7   29510110100       Missouri      St. Louis city       22.0              89.335442
8   04017942300        Arizona       Navajo County       24.0              89.226532
9   12095010400        Florida       Orange County       24.0              88

### Cell 17 — county-level candidate community seeds

In [56]:
# Cell 17 — county-level candidate community seeds

candidate_communities = (
    top_national_candidates
    .groupby(["display_state", "display_county"], dropna=False)
    .agg(
        n_shortlisted_tracts=("geoid", "nunique"),
        mean_shortlist_score=("total_shortlist_score", "mean"),
        max_shortlist_score=("total_shortlist_score", "max"),
        mean_igs_total=("igs_total", "mean"),
        mean_poverty_rate=("poverty_rate", "mean"),
        mean_insured_share=("insured_share", "mean"),
        mean_unemp_rate=("unemp_rate", "mean"),
        mean_affordable_housing_share=("affordable_housing_share", "mean"),
        total_pop=("pop_total", "sum"),
        sample_tracts=("geoid", lambda s: ", ".join(s.astype(str).head(5))),
        sample_names=("display_name", lambda s: " | ".join(s.astype(str).head(3))),
    )
    .reset_index()
    .sort_values(
        ["n_shortlisted_tracts", "mean_shortlist_score", "max_shortlist_score"],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

print(candidate_communities.head(25))

    display_state        display_county  n_shortlisted_tracts  mean_shortlist_score  max_shortlist_score  mean_igs_total  mean_poverty_rate  mean_insured_share  mean_unemp_rate  \
0     Puerto Rico    San Juan Municipio                    14             85.792400            91.089515       25.614286           0.607932            0.819181         0.203197   
1        Illinois           Cook County                     9             86.357391            88.465442       22.666667           0.397891            0.849558         0.235932   
2         Alabama         Mobile County                     8             86.916284            88.467327       22.000000           0.438018            0.824336         0.196453   
3       Tennessee         Shelby County                     7             87.296375            91.759100       22.857143           0.478255            0.823091         0.241875   
4            Ohio       Cuyahoga County                     5             86.015930            87.09

### Cell 18 — save outputs for team discussion

In [57]:
# Cell 18 — save shortlist outputs for team discussion

if SAVE_INTERMEDIATE:
    safe_save_parquet(tract_shortlist_scored, OUT_DIR / "tract_shortlist_scored.parquet")
    safe_save_parquet(top_national_candidates, OUT_DIR / "top_national_candidates.parquet")
    safe_save_parquet(state_balanced_candidates, OUT_DIR / "state_balanced_candidates.parquet")
    safe_save_parquet(candidate_communities, OUT_DIR / "candidate_communities.parquet")

tract_shortlist_scored.to_csv(OUT_DIR / "tract_shortlist_scored.csv", index=False)
top_national_candidates.to_csv(OUT_DIR / "top_national_candidates.csv", index=False)
state_balanced_candidates.to_csv(OUT_DIR / "state_balanced_candidates.csv", index=False)
candidate_communities.to_csv(OUT_DIR / "candidate_communities.csv", index=False)

print("Saved shortlist outputs to:", OUT_DIR)

Saved shortlist outputs to: data\processed


### Cell 19 — quick sanity checks before you present results to the team

In [58]:
# Cell 19 — quick sanity checks before sharing with the team

print("Top 10 states by count in national shortlist:")
print(top_national_candidates["display_state"].value_counts().head(10))

print()
print("Top 10 counties by count in national shortlist:")
print(
    top_national_candidates
    .assign(state_county=lambda d: d["display_county"].astype(str) + ", " + d["display_state"].astype(str))
    ["state_county"]
    .value_counts()
    .head(10)
)

print()
print("Tiny tract flags in national shortlist:")
print(top_national_candidates["tiny_tract_flag"].value_counts(dropna=False))

print()
print("Summary of final shortlist scores:")
print(top_national_candidates[
    ["igs_distress_score", "healthcare_proxy_score", "vulnerability_score", "total_shortlist_score"]
].describe())

Top 10 states by count in national shortlist:
display_state
Puerto Rico    37
Alabama        16
Georgia        11
Illinois       10
Louisiana       9
Tennessee       8
Mississippi     8
Ohio            6
Arizona         6
Michigan        5
Name: count, dtype: int64

Top 10 counties by count in national shortlist:
state_county
San Juan Municipio, Puerto Rico    14
Cook County, Illinois               9
Mobile County, Alabama              8
Shelby County, Tennessee            7
Bibb County, Georgia                5
Wayne County, Michigan              5
Cuyahoga County, Ohio               5
Orleans Parish, Louisiana           5
Arecibo Municipio, Puerto Rico      4
Lake County, Indiana                4
Name: count, dtype: int64

Tiny tract flags in national shortlist:
tiny_tract_flag
False    99
True     51
Name: count, dtype: int64

Summary of final shortlist scores:
       igs_distress_score  healthcare_proxy_score  vulnerability_score  total_shortlist_score
count          150.000000    